# Treinar ao vivo — com os dados que a turma gerou

Este notebook é para o **professor rodar na aula**, depois que a turma jogou o "Treine o Algoritmo".

Fluxo:
1. A turma jogou → você baixou `jogadas_coletadas.csv` (botão "Professor" no app)
2. Aqui você treina um modelo de sentimento **na frente deles**
3. Mostra a acurácia e compara com o desempenho da turma

> Se quiser treinar com a base completa (não só as jogadas), use o `reviews.csv` — explicado no fim.

## 1. Carregar os dados coletados

Coloque o `jogadas_coletadas.csv` (baixado do app) nesta pasta.

In [ ]:
import pandas as pd

# Dados que a turma gerou jogando
jogadas = pd.read_csv("jogadas_coletadas.csv")
print(f"Jogadas coletadas: {len(jogadas)}")
print(f"Alunos diferentes: {jogadas['nick'].nunique()}")
jogadas.head()

Cada linha é uma jogada: o texto do review, a **verdade** (a estrela do cliente),
o **palpite** do aluno e o que o **modelo** previu.

Para treinar um classificador, usamos: **texto → verdade** (o rótulo real).

In [ ]:
# Monta o dataset de treino: texto -> rótulo verdadeiro
dados = jogadas[["texto", "verdade"]].drop_duplicates(subset="texto").dropna()
print(f"Reviews únicos para treinar: {len(dados)}")
print(dados["verdade"].value_counts())

## 2. Treinar o modelo (ao vivo!)

Mesmo modelo do jogo: transforma texto em números (TF-IDF) e classifica (Regressão Logística).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time

X_tr, X_te, y_tr, y_te = train_test_split(
    dados["texto"], dados["verdade"], test_size=0.25, random_state=42
)

modelo = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=3000, ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

t0 = time.time()
modelo.fit(X_tr, y_tr)
print(f"Treinou em {time.time()-t0:.2f}s com {len(X_tr)} exemplos da turma")

acc = accuracy_score(y_te, modelo.predict(X_te))
print(f"Acurácia: {acc*100:.1f}%")

> **Conversa com a turma:** com poucos exemplos a acurácia oscila bastante.
> Pergunte: "se a gente jogar mais 10 minutos, vocês acham que melhora?"
> (Sim — mais dados de treino quase sempre ajudam, até certo ponto.)

## 3. Testar o modelo recém-treinado

In [ ]:
for frase in [
    "Produto maravilhoso, chegou super rápido!",
    "Veio com defeito e não consigo trocar",
    "demorou demais mas no fim deu certo",
]:
    p = modelo.predict_proba([frase])[0][1]
    rotulo = "👍 Satisfeito" if p >= 0.5 else "👎 Insatisfeito"
    print(f"{rotulo} ({p:.0%})  ←  {frase}")

## 4. Como a turma se saiu vs o modelo?

In [ ]:
# Acurácia média da turma (dos dados coletados)
acc_turma = jogadas["acertou"].mean()
acc_modelo_jogo = jogadas["modelo_acertou"].mean()

print(f"Turma (média):         {acc_turma*100:.1f}%")
print(f"Modelo (durante jogo): {acc_modelo_jogo*100:.1f}%")
print(f"Modelo (treinado agora): {acc*100:.1f}%")

## 5. (Opcional) Treinar com a base completa

Se a turma gerou poucos dados, treine com o `reviews.csv` (6.000 reviews) para mostrar
o efeito de **mais dados**.

In [ ]:
base = pd.read_csv("reviews.csv")
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    base["texto"], base["positivo"], test_size=0.25, random_state=42, stratify=base["positivo"]
)
modelo_grande = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
]).fit(Xb_tr, yb_tr)

acc_grande = accuracy_score(yb_te, modelo_grande.predict(Xb_te))
print(f"Com {len(Xb_tr)} exemplos: {acc_grande*100:.1f}%")
print(f"(vs {acc*100:.1f}% com os {len(X_tr)} exemplos da turma)")
print("\nMais dados → modelo melhor e mais estável.")

## Próxima aula

Na **Aula 4**, vamos pegar exatamente este problema (classificar sentimento do texto)
e resolver com um **agente de IA / LLM** — que faz isso **sem treinar nada**, só lendo.

A pergunta: será que o LLM bate este modelo que vocês treinaram hoje? 🚀